In [1]:
#%reload_ext autoreload
#%autoreload 2
%matplotlib inline

In [2]:
! pip install -q transformers[sentencepiece] datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import transformers

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%cd /content/drive/MyDrive/udemy_project/Text Classification Huggingface Transformers

/content/drive/MyDrive/udemy_project/Text Classification Huggingface Transformers


URL to the model: https://huggingface.co/distilroberta-base

In [6]:
model_name = "distilroberta-base"

#Data

In [7]:
df = pd.read_csv("/content/drive/MyDrive/udemy_project/Datasets/course_details.csv")
df.head()

,title,url,description,topic,topic_list
0,The Complete Python Bootcamp From Zero to Hero...,https://www.udemy.com/course/complete-python-b...,Become a Python Programmer and learn one of em...,Python,"['Development', 'Programming Languages', 'Pyth..."
1,The Complete Full-Stack Web Development Bootcamp,https://www.udemy.com/course/the-complete-web-...,Welcome to the Complete Web Development Bootca...,Web Development,"['Development', 'Web Development', 'Node.Js', ..."
2,100 Days of Code™: The Complete Python Pro Boo...,https://www.udemy.com/course/100-days-of-code,Welcome to the 100 Days of Code - The Complete...,Python,"['Development', 'Programming Languages', 'Pyth..."
3,The Web Developer Bootcamp 2026,https://www.udemy.com/course/the-web-developer...,Now with over 10 hours of React content. Massi...,Web Development,"['Development', 'Web Development']"
4,"React - The Complete Guide (incl. Next.js, Redux)",https://www.udemy.com/course/react-the-complet...,"This bestselling course by the author of ""Reac...",React JS,"['Development', 'Programming Languages', 'Reac..."


In [8]:
df = df.dropna().reset_index(drop=True)


In [9]:
topic_list = []
indices_to_drop = []
for idx, topics in enumerate(df.topic_list.to_list()):
  topics_list = eval(topics)
  if len(topics_list):
    topic_list.append(topics_list[0])
  else:
    indices_to_drop.append(idx)

df = df.drop(indices_to_drop).reset_index(drop=True)
df.shape

(11057, 5)

In [10]:
df['primary_topic'] = topic_list

In [11]:
topic_count = df['primary_topic'].value_counts()
threshold = int(len(df) * 0.005)
rare_topics = [cat for cat, count in topic_count.items() if count < threshold]
len(rare_topics), rare_topics[:5]

(3, ['Udemy Free Resource Center', 'Vodafone', 'Health & Fitness'])

In [12]:
rare_indices_to_drop = [idx for idx, row in df.iterrows() if row['primary_topic'] in rare_topics]
len(rare_indices_to_drop)

16

In [13]:
df = df.drop(rare_indices_to_drop).reset_index(drop=True)
df.shape

(11041, 6)

In [14]:
df['primary_topic'].value_counts()

,count
primary_topic,
Development,1597
Finance & Accounting,1586
Business,1583
Marketing,1583
Personal Development,1577
IT & Software,1561
Design,1554


In [15]:
len(df['primary_topic'].value_counts())

7

In [16]:
labels = list(set(df.primary_topic.to_list()))
label_count = len(labels)
labels, label_count

(['Design',
  'IT & Software',
  'Business',
  'Finance & Accounting',
  'Development',
  'Personal Development',
  'Marketing'],
 7)

In [17]:
df.head()

,title,url,description,topic,topic_list,primary_topic
0,The Complete Python Bootcamp From Zero to Hero...,https://www.udemy.com/course/complete-python-b...,Become a Python Programmer and learn one of em...,Python,"['Development', 'Programming Languages', 'Pyth...",Development
1,The Complete Full-Stack Web Development Bootcamp,https://www.udemy.com/course/the-complete-web-...,Welcome to the Complete Web Development Bootca...,Web Development,"['Development', 'Web Development', 'Node.Js', ...",Development
2,100 Days of Code™: The Complete Python Pro Boo...,https://www.udemy.com/course/100-days-of-code,Welcome to the 100 Days of Code - The Complete...,Python,"['Development', 'Programming Languages', 'Pyth...",Development
3,The Web Developer Bootcamp 2026,https://www.udemy.com/course/the-web-developer...,Now with over 10 hours of React content. Massi...,Web Development,"['Development', 'Web Development']",Development
4,"React - The Complete Guide (incl. Next.js, Redux)",https://www.udemy.com/course/react-the-complet...,"This bestselling course by the author of ""Reac...",React JS,"['Development', 'Programming Languages', 'Reac...",Development


In [18]:
df.describe(include='object')

,title,url,description,topic,topic_list,primary_topic
count,11041,11041,11041,11041,11041,11041
unique,11021,11041,11033,1959,8303,7
top,HTML & CSS - Certification Course for Beginners,https://www.udemy.com/course/professional-ethi...,We live in an instant gratification culture to...,Python,"['Development', 'Programming Languages', 'Pyth...",Development
freq,2,1,3,133,47,1597


# Data Processing

In [19]:
from datasets import Dataset, DatasetDict
ds = Dataset.from_pandas(df)


In [20]:
ds

Dataset({
    features: ['title', 'url', 'description', 'topic', 'topic_list', 'primary_topic'],
    num_rows: 11041
})

In [21]:
ds[0]

{'title': 'The Complete Python Bootcamp From Zero to Hero in Python',
 'url': 'https://www.udemy.com/course/complete-python-bootcamp',
 'description': "Become a Python Programmer and learn one of employer's most requested skills of 2023! This is the most comprehensive, yet straight-forward, course for the Python programming language on Udemy! Whether you have never programmed before, already know basic syntax, or want to learn about the advanced features of Python, this course is for you! In this course we will teach you Python 3. With over 100 lectures and more than 21 hours of video this comprehensive course leaves no stone unturned! This course includes quizzes, tests, coding exercises and homework assignments as well as 3 major projects to create a Python project portfolio! Learn how to use Python for real-world tasks, such as working with PDF Files, sending emails, reading Excel files, Scraping websites for informations, working with image files, and much more! This course will te

## Tokenization

In [25]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [26]:
tokenizer.tokenize(ds[0]["description"][:150])

['Bec',
 'ome',
 'Ġa',
 'ĠPython',
 'ĠProgram',
 'mer',
 'Ġand',
 'Ġlearn',
 'Ġone',
 'Ġof',
 'Ġemployer',
 "'s",
 'Ġmost',
 'Ġrequested',
 'Ġskills',
 'Ġof',
 'Ġ20',
 '23',
 '!',
 'ĠThis',
 'Ġis',
 'Ġthe',
 'Ġmost',
 'Ġcomprehensive',
 ',',
 'Ġyet',
 'Ġstraight',
 '-',
 'forward',
 ',',
 'Ġcourse',
 'Ġfor']

In [27]:
tokenizer(ds[0]['description'][:150])

{'input_ids': [0, 41094, 4399, 10, 31886, 4928, 2089, 8, 1532, 65, 9, 8850, 18, 144, 5372, 2417, 9, 291, 1922, 328, 152, 16, 5, 144, 5145, 6, 648, 1359, 12, 16135, 6, 768, 13, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [28]:
def tokenize_desc(x) :
  return tokenizer(x['description'], truncation=True, padding=True)

In [29]:
tokenized_ds = ds.map(tokenize_desc, batched=True)

Map:   0%|          | 0/11041 [00:00<?, ? examples/s]

In [30]:
tokenized_ds[0]

{'title': 'The Complete Python Bootcamp From Zero to Hero in Python',
 'url': 'https://www.udemy.com/course/complete-python-bootcamp',
 'description': "Become a Python Programmer and learn one of employer's most requested skills of 2023! This is the most comprehensive, yet straight-forward, course for the Python programming language on Udemy! Whether you have never programmed before, already know basic syntax, or want to learn about the advanced features of Python, this course is for you! In this course we will teach you Python 3. With over 100 lectures and more than 21 hours of video this comprehensive course leaves no stone unturned! This course includes quizzes, tests, coding exercises and homework assignments as well as 3 major projects to create a Python project portfolio! Learn how to use Python for real-world tasks, such as working with PDF Files, sending emails, reading Excel files, Scraping websites for informations, working with image files, and much more! This course will te

In [31]:
row = tokenized_ds[0]
row['description'], row['input_ids']

("Become a Python Programmer and learn one of employer's most requested skills of 2023! This is the most comprehensive, yet straight-forward, course for the Python programming language on Udemy! Whether you have never programmed before, already know basic syntax, or want to learn about the advanced features of Python, this course is for you! In this course we will teach you Python 3. With over 100 lectures and more than 21 hours of video this comprehensive course leaves no stone unturned! This course includes quizzes, tests, coding exercises and homework assignments as well as 3 major projects to create a Python project portfolio! Learn how to use Python for real-world tasks, such as working with PDF Files, sending emails, reading Excel files, Scraping websites for informations, working with image files, and much more! This course will teach you Python in a practical manner, with every lecture comes a full coding screencast and a corresponding code notebook! Learn in whatever manner is

In [32]:
# Vocabulary index, Numericalization like we did in ULMFit
tokenizer.vocab['Ġunforgettable']

26865

## Categorize

In [33]:
labels

['Design',
 'IT & Software',
 'Business',
 'Finance & Accounting',
 'Development',
 'Personal Development',
 'Marketing']

In [35]:
labels.index('Marketing')

6

In [40]:
def categorize(x):
  return {"labels": [labels.index(genre) for genre in x['primary_topic']]}

In [41]:
categorized_ds = tokenized_ds.map(categorize, batched=True)
categorized_ds

Map:   0%|          | 0/11041 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'url', 'description', 'topic', 'topic_list', 'primary_topic', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 11041
})

In [42]:
categorized_ds[0]

{'title': 'The Complete Python Bootcamp From Zero to Hero in Python',
 'url': 'https://www.udemy.com/course/complete-python-bootcamp',
 'description': "Become a Python Programmer and learn one of employer's most requested skills of 2023! This is the most comprehensive, yet straight-forward, course for the Python programming language on Udemy! Whether you have never programmed before, already know basic syntax, or want to learn about the advanced features of Python, this course is for you! In this course we will teach you Python 3. With over 100 lectures and more than 21 hours of video this comprehensive course leaves no stone unturned! This course includes quizzes, tests, coding exercises and homework assignments as well as 3 major projects to create a Python project portfolio! Learn how to use Python for real-world tasks, such as working with PDF Files, sending emails, reading Excel files, Scraping websites for informations, working with image files, and much more! This course will te

In [43]:
row = categorized_ds[0]
row['labels']

4

# Data Splitting

In [44]:
split_ds = categorized_ds.train_test_split(0.1, seed=42)
split_ds

DatasetDict({
    train: Dataset({
        features: ['title', 'url', 'description', 'topic', 'topic_list', 'primary_topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 9936
    })
    test: Dataset({
        features: ['title', 'url', 'description', 'topic', 'topic_list', 'primary_topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1105
    })
})

# Modeling

In [45]:
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

In [46]:
bs = 32
epochs = 7
lr = 3.75e-4

In [50]:
# args = TrainingArguments(
#     "models",
#     learning_rate = lr,
#     warmup_ratio = 0.1,
#     lr_scheduler_type='cosine',
#     fp16=True,
#     evaluation_strategy='epoch',
#     per_device_train_batch_size=bs,
#     per_device_eval_batch_size=bs,
#     num_train_epochs=epochs,
#     weight_decay=0.01,
#     report_to='none',
#     # save_steps=200,
#     # save_strategy='epochs'
# )
args = TrainingArguments(
    "models",
    learning_rate=lr,
    warmup_steps=0.1,       # ratio-as-float, replaces old warmup_ratio
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy='epoch',  # renamed from evaluation_strategy
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none',
)

In [51]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=label_count)
model

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [52]:
import evaluate
import numpy as np
def accuracy(eval_preds):
  metric = evaluate.load("accuracy")
  logits, labels = eval_preds
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)

In [55]:
trainer = Trainer(
    model,
    args,
    train_dataset=split_ds['train'],
    eval_dataset=split_ds['test'],
    processing_class=tokenizer,   # renamed from tokenizer=
    compute_metrics=accuracy
)

In [56]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.158160,0.592760
2,0.925859,0.957443,0.646154
3,0.925859,0.982050,0.680543
4,0.875800,0.863469,0.695928
5,0.749205,0.773172,0.695023
6,0.749205,0.789225,0.720362
7,0.598150,0.764073,0.758371


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2177, training_loss=0.7671747974260827, metrics={'train_runtime': 1007.0718, 'train_samples_per_second': 69.064, 'train_steps_per_second': 2.162, 'total_flos': 9214194048417792.0, 'train_loss': 0.7671747974260827, 'epoch': 7.0})

In [57]:
trainer.save_model('models/course-classifier')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]